In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE_PATH = "/Volumes/property_catalog/property_management/landing_zone/properties.csv"
TARGET_SCHEMA = "property_catalog.property_management"

RAW_TABLE = f"{TARGET_SCHEMA}.raw_properties"
CLEAN_TABLE = f"{TARGET_SCHEMA}.clean_properties"
REJECTED_TABLE = f"{TARGET_SCHEMA}.rejected_properties"

In [0]:
def extract_properties(source_path):
    properties_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(source_path)
    )

    print("Properties extracted successfully!")

    # Log the number of source records
    print(f"Properties extracted: {properties_df.count()} rows")

    return properties_df

properties_df = extract_properties(SOURCE_PATH)

# Save raw properties to a managed Delta table
(
    properties_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(RAW_TABLE)
)

display(properties_df)

In [0]:
def transform_properties(properties_df):
    # Standardise property columns and data types
    transformed_df = (
        properties_df
        .withColumn("Id", F.expr("try_cast(trim(Id) AS BIGINT)"))
        .withColumn("Address", F.trim(F.col("Address")))
        .withColumn("Rent", F.expr("try_cast(Rent AS DOUBLE)"))
        .withColumn("Status", F.initcap(F.trim(F.col("Status"))))
        .withColumn("LandlordId", F.trim(F.col("LandlordId")))
    )

    print("Properties transformation completed!")
    print(f"Total records: {transformed_df.count()}")

    display(transformed_df)

    return transformed_df

# Read raw properties Delta table
properties_df = spark.table(RAW_TABLE)

transformed_properties_df = transform_properties(properties_df)

In [0]:
def validate_properties(transformed_properties_df):
    validated_df = (
        transformed_properties_df
        
        # Flag duplicate property IDs
        .withColumn("duplicate_property_id", F.count("*").over(Window.partitionBy("Id")) > 1)

        # Flag records with missing required values
        .withColumn(
            "has_missing_values",
            F.col("Id").isNull()
            | F.col("Address").isNull()
            | (F.col("Address") == "")
            | F.col("Rent").isNull()
            | F.col("Status").isNull()
            | F.col("LandlordId").isNull()
        )

        # Flag missing or negative rent
        .withColumn("invalid_rent", F.col("Rent").isNull() | (F.col("Rent") < 0))

        # Flag missing, empty or over 200 character addresses
        .withColumn(
            "invalid_address",
            F.col("Address").isNull()
            | (F.col("Address") == "")
            | (F.length("Address") > 200)
        )

        # Flag missing or unrecognised property status
        .withColumn("invalid_status", F.col("Status").isNull() | (~F.col("Status").isin("Occupied", "Vacant")))
    )

    print("Property validation completed!")

    return validated_df

validated_properties_df = validate_properties(transformed_properties_df)

display(validated_properties_df)

In [0]:
def separate_properties(validated_properties_df):
    rejection_condition = (
        F.col("has_missing_values")
        | F.col("duplicate_property_id")
        | F.col("invalid_rent")
        | F.col("invalid_address")
        | F.col("invalid_status")
    )

    # Filter rejected property records and add detailed reasons for each validation failure
    rejected_properties_df = (
        validated_properties_df
        .filter(rejection_condition)
        .withColumn(
            "rejection_reason",
            F.concat_ws(
                ". ",
                F.when(F.col("has_missing_values"), F.lit("Missing required values")),
                F.when(F.col("duplicate_property_id"), F.lit("Duplicate property ID")),
                F.when(F.col("invalid_rent"), F.lit("Rent is missing or negative")),
                F.when(F.col("invalid_address"), F.lit("Address is missing or exceeds 200 characters")),
                F.when(F.col("invalid_status"), F.lit("Status must be Occupied or Vacant"))
            )
        )
    )

    # Filter clean property records
    clean_properties_df = validated_properties_df.filter(~rejection_condition)

    return clean_properties_df, rejected_properties_df

clean_properties_df, rejected_properties_df = separate_properties(validated_properties_df)

print("Clean properties:")
display(clean_properties_df)

print("Rejected properties:")
display(rejected_properties_df)

In [0]:
def summarise_properties(properties_df, clean_properties_df, rejected_properties_df):
    total_count = properties_df.count()
    clean_count = clean_properties_df.count()
    rejected_count = rejected_properties_df.count()

    # Calculate the clean record success rate
    success_rate = (
        clean_count / total_count * 100
        if total_count > 0
        else 0
    )

    print("=" * 45)
    print("PROPERTIES ETL SUMMARY")
    print("=" * 45)
    print(f"Source: {SOURCE_PATH}")
    print(f"Total records: {total_count}")
    print(f"Clean records: {clean_count}")
    print(f"Rejected records: {rejected_count}")
    print(f"Success rate: {success_rate:.2f}%")
    print("=" * 45)

    return {
        "total": total_count,
        "clean": clean_count,
        "rejected": rejected_count,
        "success_rate": success_rate
    }

properties_summary = summarise_properties(properties_df, clean_properties_df, rejected_properties_df)

In [0]:
def load_properties(clean_properties_df, rejected_properties_df):
    # Remove validation flags from the clean output
    clean_properties_output_df = clean_properties_df.drop(
        "duplicate_property_id",
        "has_missing_values",
        "invalid_rent",
        "invalid_address",
        "invalid_status"
    )

    # Save clean properties to a managed Delta table
    (
        clean_properties_output_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(CLEAN_TABLE)
    )

    # Save rejected properties to a managed Delta table
    (
        rejected_properties_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(REJECTED_TABLE)
    )

    print(f"Clean property records saved to: {CLEAN_TABLE}")
    print(f"Rejected property records saved to: {REJECTED_TABLE}")


load_properties(clean_properties_df, rejected_properties_df)